# 第五章 大数据分析策略及机器学习
----
书籍名称：《大数据分析：Python 实践与大型语言模型应用》

谢佳松、刘娟



----

## 5.1 三类统计分析策略

### 5.1.3 探索性统计分析

In [ ]:
#T检验

In [ ]:
#读取第三方包
from scipy import stats
 
#读取充电宝容量数据
volumns =[4988, 5006, 5021, 4923, 4947, 4896, 5104, 4992, 5070, 5009, 4892, 4997]
 
#单样本t检验
stats.ttest_1samp(a=volumns, popmean=5000)

In [ ]:
#卡方检验
from scipy.stats import chi2_contingency

# 构建观察频数表
observed = [[50, 40, 30],
            [70, 60, 50]]

# 执行卡方检验
chi2, p, dof, expected = chi2_contingency(observed)

print(f"卡方统计量: {chi2:.4f}")
print(f"P值: {p:.4f}")
print(f"自由度: {dof}")
print("期望频数表:")
print(expected)

In [ ]:
#皮尔逊相关性检验
from scipy.stats import pearsonr

# 学习时间(小时)
study_time = [5, 7, 3, 1, 9, 6, 4, 2, 8, 10]

# 考试成绩(分)
exam_scores = [60, 75, 50, 40, 85, 70, 55, 45, 80, 90]

# 计算皮尔逊相关系数和P值
corr, p_value = pearsonr(study_time, exam_scores)

print(f"皮尔逊相关系数: {corr:.4f}")
print(f"P值: {p_value:.6f}")

In [ ]:
# Shapiro 正态性检验
from scipy import stats
import numpy as np
try:
    import seaborn as sns
    titanic = sns.load_dataset("titanic")
    cleaned_age = titanic["age"].dropna()
except Exception:
    # 若 seaborn 或在线数据不可用，则退回到本地模拟样本
    rng = np.random.default_rng(42)
    cleaned_age = rng.normal(loc=30, scale=14, size=300)
stats.shapiro(cleaned_age)


## 5.2 时间序列分析

### 5.2.1 时间序列分析基础 

In [ ]:
# 加载必要库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

### 5.2.2 获取数据

In [ ]:
# 获取时间序列数据
import pandas as pd
try:
    from statsmodels.datasets import get_rdataset
    airpass = get_rdataset("AirPassengers")
    data = airpass.data.set_index("time")
    data.columns = ["Passengers"]
    data.index = pd.date_range(start="1949-01-01", periods=len(data), freq="M")
except Exception:
    passengers = [112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432]
    data = pd.DataFrame({"Passengers": passengers}, index=pd.date_range(start="1949-01-01", periods=144, freq="M"))
print("数据前5行:")
print(data.head())
print("\n数据类型:")
print(data.dtypes)


### 5.2.3 绘制时间序列图

In [ ]:
plt.figure(figsize=(10,8), dpi=1000)
data.plot()
plt.title('国际航空乘客数量 (1949-1960)')
plt.ylabel('乘客数量 (千人)')
plt.xlabel('日期')
plt.grid(True)
plt.tight_layout()
plt.savefig('img/国际航空乘客数量1949-1960.pdf')
plt.show()

### 5.2.4 自相关

In [ ]:
plt.figure(figsize=(12, 6), dpi=5000)
plot_acf(data, lags=40)
plt.title('原始序列自相关图')
plt.tight_layout()
plt.savefig('img/原始序列自相关图.pdf')
plt.show()

### 5.2.5 平稳性检验

In [ ]:
def adf_test(series, title=''):
    """执行ADF检验并输出结果"""
    result = adfuller(series, autolag='AIC')
    print(f'ADF检验: {title}')
    print(f'ADF统计量: {result[0]:.4f}')
    print(f'p值: {result[1]:.4f}')
    print('临界值:')
    for key, value in result[4].items():
        print(f'\t{key}: {value:.4f}')
   
# 执行ADF检验
adf_test(data['Passengers'], '原始序列')

In [ ]:
# 一阶差分
data_diff = data.diff().dropna()
data_diff.columns = ['Passengers_Diff']

# 可视化差分结果
plt.figure(figsize=(12, 6))
data_diff.plot()
plt.title('一阶差分序列')
plt.tight_layout()
plt.savefig('img/一阶差分序列.pdf')
plt.show()

# 一阶差分ADF检验
adf_test(data_diff['Passengers_Diff'], '一阶差分序列')

In [ ]:
# 二阶差分
data_diff2 = data_diff.diff().dropna()
data_diff2.columns = ['Passengers_Diff2']

# 可视化比较
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
data_diff.plot(ax=axes[0])
axes[0].set_title('一阶差分')
data_diff2.plot(ax=axes[1])
axes[1].set_title('二阶差分')
plt.tight_layout()
plt.savefig('img/一阶二阶差分.pdf')
plt.show()
# 二阶差分ADF检验
adf_test(data_diff2['Passengers_Diff2'], '二阶差分序列')

### 5.2.7 合适的p，q

In [ ]:
# 绘制差分序列ACF和PACF图
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(data_diff, lags=40, ax=axes[0])
axes[0].set_title('差分序列自相关图(ACF)')
plot_pacf(data_diff, lags=40, ax=axes[1])
axes[1].set_title('差分序列偏自相关图(PACF)')
plt.tight_layout()
plt.savefig('img/差分序列自相关图.pdf')
plt.show()

In [ ]:
# 如需自动寻参，可先安装 pmdarima
%pip install pmdarima


In [ ]:
# 使用 auto_arima 自动选择最佳参数；若本机未安装 pmdarima，则回退为固定阶数示例
try:
    from pmdarima import auto_arima
    model = auto_arima(data["Passengers"], seasonal=True, m=12, trace=True, error_action="ignore", suppress_warnings=True, stepwise=True)
    print(model.summary())
except Exception as e:
    print("pmdarima 不可用，回退为固定阶数示例:", e)
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    model = SARIMAX(data["Passengers"], order=(1,1,1), seasonal_order=(1,1,1,12))
    results = model.fit(disp=False)
    print(results.summary())


### 5.2.8 模型检验

In [ ]:
# 拆分训练集和测试集
train = data.iloc[:-12]  # 保留最后12个月作为测试
test = data.iloc[-12:]
 
# 拟合SARIMA模型
from statsmodels.tsa.statespace.sarimax import SARIMAX
 
model = SARIMAX(train, 
                order=(1, 1, 1), 
                seasonal_order=(1, 1, 1, 12))
results = model.fit()
 
# 输出模型摘要
print(results.summary()) 

In [ ]:
# 残差诊断图
results.plot_diagnostics(figsize=(15, 12))
plt.tight_layout()
plt.show()
 
# Ljung-Box检验
from statsmodels.stats.diagnostic import acorr_ljungbox
 
lb_test = acorr_ljungbox(results.resid, lags=[10, 20, 30], return_df=True)
print("Ljung-Box检验结果:")
print(lb_test) 

### 5.2.9 模型预测

In [ ]:
#预测
# 预测未来12个月
forecast = results.get_forecast(steps=12)
forecast_mean = forecast.predicted_mean
conf_int = forecast.conf_int()
 
# 可视化预测结果
plt.figure(figsize=(14, 7))
plt.plot(data.index, data['Passengers'], label='实际数据')
plt.plot(forecast_mean.index, forecast_mean, color='red', label='预测值')
plt.fill_between(conf_int.index, 
                 conf_int.iloc[:, 0], 
                 conf_int.iloc[:, 1], 
                 color='pink', alpha=0.3, label='95%置信区间')
plt.title('航空乘客数量预测')
plt.xlabel('日期')
plt.ylabel('乘客数量')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('img/航空乘客数据预测.pdf')
plt.show()

In [ ]:
# 评估预测准确性
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
 
# 计算评估指标
mae = mean_absolute_error(test, forecast_mean)
mape = mean_absolute_percentage_error(test, forecast_mean) * 100
 
print(f'预测性能评估:')
print(f'平均绝对误差(MAE): {mae:.2f}')
print(f'平均绝对百分比误差(MAPE): {mape:.2f}%')
 
# 预测值与实际值对比
comparison = pd.DataFrame({
    '实际值': test['Passengers'],
    '预测值': forecast_mean,
    '相对误差(%)': np.abs((test['Passengers'].values - forecast_mean.values) / test['Passengers'].values) * 100
})
print('\n预测值与实际值对比:')
print(comparison)

## 5.3 机器学习及 sklearn 库

## 5.4 线性回归

### 5.4.1 使用 sklearn 实现房价预测

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
import sklearn
 
# d = datasets.load_boston() #新版本中已删除该数据集
# https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data   数据下载地址
feature_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
    'PTRATIO', 'B', 'LSTAT','PRICE'
]

# 方法1：直接从UCI网站下载数据（推荐）
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data"
boston = pd.read_csv("data/housing_boston.csv")

# 方法2：使用本地文件（确保路径正确）
# file_path = '/Users/.../housing_boston.csv'
# boston = pd.read_csv(file_path, sep='\s+', header=None, names=feature_names)

# 显示数据集信息
boston.info()

In [ ]:
boston.keys()

In [ ]:
boston.head()

In [ ]:
X = boston.drop('PRICE', axis = 1) 
y = boston['PRICE']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [ ]:
print(X_train, shape)
print(X_train, test)
print(y_train, shape)
print(y_train, test)

In [ ]:
from sklearn.linear_model import LinearRegression
LR = LinearRegression()
LR.fit(X_train,y_train)
y_pred = LR.predict(X_test)

In [ ]:
print('w0 = ', LR.intercept_)
print('w = ',LR.coef_)

In [ ]:
np.set_printoptions(precision = 3, suppress = True)
print('w0 = {0:.3f}'.format(LR.intercept_))
print('w = {}'.format(LR.coef_))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
 
plt.scatter(y_test, y_pred)
plt.xlabel("price: $Y_i$")
plt.ylabel("Predicted prices: $\hat{Y} i$")
plt.title("Prices vs Predicted prices: $y_i$ vs $\hat{y}_i$")
plt.grid()
 
x = np.arange(0, 50)
y = x
 
plt.plot(x, y, color = 'red', lw = 4)
plt.text(30, 40,"predict line")
plt.savefig("img/price.eps")
plt.savefig('img/Price vs predicted prices.pdf')
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
 
#构造 DataFrame 数据集
data = pd.concat([pd.Series(y_test.values), pd.Series(y_pred)], axis = 1)
data.columns = ['实际房价', '预测房价']
 
#解决中文显示问题
plt.rcParams['font.sans-serif']=['Arial Unicode MS']
sns.lmplot(x = '实际房价', y='预测房价', data = data)

plt.savefig('img/实际房价.pdf')
plt.show()

In [ ]:
from sklearn import metrics
mse = metrics.mean_squared_error(y_test, y_pred)
print(f"均方误差（mse）:{mse:.4f}")

In [ ]:
df = pd.DataFrame({'实际房价': y_test, '预测房价': y_pred})
df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
import matplotlib.pyplot as plt

#(1)导入数据
feature_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
    'PTRATIO', 'B', 'LSTAT','PRICE'
]
 
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data"
#读取数据并添加列表
boston = pd.read_csv("data/housing_boston.csv")
 
#(2)分割数据04
X = boston.drop('PRICE', axis=1)
y = boston['PRICE']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size =0.2, random_state=42
)
 
#(3)导入线性回归模型并训练模型
LR = LinearRegression()
LR.fit (X_train, y_train)
 
#(4)在测试集上预测
y_pred = LR.predict (X_test)
 
#(5)评估模型
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)  # 均方根误差
mae = metrics.mean_absolute_error(y_test, y_pred)
r2 = metrics.r2_score(y_test, y_pred)

print("\n模型性能评估:")
print(f"均方误差 (MSE): {mse:.4f}")
print(f"均方根误差 (RMSE): {rmse:.4f}")
print(f"平均绝对误差 (MAE): {mae:.4f}")
print(f"决定系数 (R²): {r2:.4f}")

#(6)可视化预测结果
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color='blue', edgecolor='k')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 
         color='red', linestyle='--', lw=2, label='理想预测线')
plt.xlabel("实际价格 ($Y_i$)", fontsize=12)
plt.ylabel("预测价格 ($\hat{Y}_i$)", fontsize=12)
plt.title("实际价格 vs 预测价格", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.text(5, 45, f"R² = {r2:.4f}", fontsize=12, 
         bbox=dict(facecolor='white', alpha=0.8))
plt.tight_layout()
plt.savefig("img/price_prediction.png", dpi=300)
plt.show()

# (7) 残差分析
residuals = y_test - y_pred

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('预测价格')
plt.ylabel('残差')
plt.title('残差图')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=30, color='skyblue', edgecolor='black')
plt.xlabel('残差')
plt.ylabel('频数')
plt.title('残差分布')
plt.grid(True)

plt.tight_layout()
plt.savefig("img/residual_analysis.png", dpi=300)
plt.show()

### 补充：线性回归扩展（拓展阅读：多项式/岭/LASSO回归）

#### 多项式回归

In [ ]:
# 数据准备
import numpy as np
import matplotlib.pyplot as plt
x= np.random.uniform(-3, 3, size=100)   #产生100个随机数
X=x.reshape(-1,1)      #将x变成矩阵，一行一行的形式
y = 0.5 * x ** 2 + x + 2 + np.random.normal(0, 1, size=100)
#在数据中引入噪声
plt.scatter(x, y)
plt.savefig('img/多项式回归散点图.png', dpi=300)
plt.show( )

In [ ]:
#线性回归
from sklearn.linear_model import LinearRegression
#线性回归
lin_reg = LinearRegression()
lin_reg.fit(X, y)
y_predict = lin_reg.predict(X)
plt.rcParams['font.family']=['Arial Unicode MS']
plt.rcParams['axes.unicode_minus']=False
plt.title('线性回归')
plt.scatter(x,y)
plt.plot(x, y_predict, color = 'r')
plt.savefig('img/线性回归散点图.png', dpi=300)
plt.show()

In [ ]:
# 多项式回归
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

poly = PolynomialFeatures(degree=2)

#设置最多添加几次幂的特征项
poly.fit(X)
x2 = poly.transform(X)

#创建并训练多项式回归模型
lin_reg2 = LinearRegression()
lin_reg2.fit(x2, y)
y_predict2 = lin_reg2.predict(x2)

plt.scatter(x, y)
#绘制多项式回归曲线
plt.plot(np.sort(x), y_predict2[np.argsort(x)], color ='r')
plt.title('多项式回归')
plt.savefig('img/多项式回归散点图.png', dpi=300)
plt.show()

In [ ]:
# 多项式回归
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

poly = PolynomialFeatures(degree=20)

#设置最多添加几次幂的特征项
poly.fit(X)
x2 = poly.transform(X)

#创建并训练多项式回归模型
lin_reg2 = LinearRegression()
lin_reg2.fit(x2, y)
y_predict2 = lin_reg2.predict(x2)

plt.scatter(x, y)
#绘制多项式回归曲线
plt.plot(np.sort(x), y_predict2[np.argsort(x)], color ='r')
plt.title('多项式回归')
plt.show()

#### 岭回归

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import fetch_california_housing as fch
from sklearn.metrics import r2_score

# 加载数据集
house_value = fch()
X = pd.DataFrame(house_value.data)
y = house_value.target
X.columns = ["住户收入中位数", "房屋使用年代中位数", "平均房间数目", "平均卧室数目", 
             "街区人口", "平均入住率", "街区的纬度", "街区的经度"]

# 创建包含特征和目标的数据框
Xtmp = X.copy()  # 创建副本而不是引用
Xtmp['价格'] = y
display(Xtmp.head())  # 显示前几行而不是整个数据集

# 划分训练集、测试集
xtrain, xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.3, random_state=420
)

# 训练集和测试集恢复索引
xtrain.index = range(xtrain.shape[0])
xtest.index = range(xtest.shape[0])

# 采用岭回归训练模型
reg = Ridge(alpha=5).fit(xtrain, ytrain)

# 在测试集上进行预测并计算R²
yhat = reg.predict(xtest)
r2 = r2_score(ytest, yhat)  # 修正参数顺序：先真实值，后预测值
print("测试集R²: %.8f" % r2)

# 探索交叉验证下岭回归与线性回归的结果变化
alpha_range = np.arange(1, 1001, 100)
ridge_scores, lr_scores = [], []

# 线性回归模型只需计算一次（不依赖alpha）
linear = LinearRegression()
linear_score = cross_val_score(linear, X, y, cv=5, scoring='r2').mean()

for alpha in alpha_range:
    # 岭回归模型
    reg = Ridge(alpha=alpha)
    ridge_score = cross_val_score(reg, X, y, cv=5, scoring='r2').mean()
    ridge_scores.append(ridge_score)
    
    # 线性回归分数是常数，不需要重复计算
    lr_scores.append(linear_score)

# 绘制结果
plt.figure(figsize=(12, 6))
plt.plot(alpha_range, ridge_scores, c='red', label='岭回归 (Ridge)')
plt.plot(alpha_range, lr_scores, c='orange', label='线性回归 (LR)')
plt.title('不同正则化强度下的模型性能')
plt.xlabel('正则化强度 (alpha)')
plt.ylabel('交叉验证R²')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# 标记最佳alpha值
best_alpha = alpha_range[np.argmax(ridge_scores)]
best_score = max(ridge_scores)
plt.axvline(x=best_alpha, color='blue', linestyle=':', 
            label=f'最佳alpha={best_alpha}\nR²={best_score:.4f}')
plt.legend()

plt.show()

# 使用最佳alpha值重新训练模型
print(f"\n使用最佳alpha值({best_alpha})重新训练模型...")
best_reg = Ridge(alpha=best_alpha).fit(xtrain, ytrain)

# 评估最佳模型
train_pred = best_reg.predict(xtrain)
test_pred = best_reg.predict(xtest)

train_r2 = r2_score(ytrain, train_pred)
test_r2 = r2_score(ytest, test_pred)

print(f"训练集R²: {train_r2:.6f}")
print(f"测试集R²: {test_r2:.6f}")

# 比较特征重要性
plt.figure(figsize=(12, 6))
features = X.columns

# 线性回归系数
lr_coef = LinearRegression().fit(xtrain, ytrain).coef_

# 岭回归系数
ridge_coef = best_reg.coef_

# 绘制系数比较
x = np.arange(len(features))
width = 0.35

plt.bar(x - width/2, lr_coef, width, label='线性回归')
plt.bar(x + width/2, ridge_coef, width, label=f'岭回归 (alpha={best_alpha})')
plt.axhline(0, color='black', linewidth=0.8)

plt.xlabel('特征')
plt.ylabel('系数值')
plt.title('线性回归与岭回归特征系数比较')
plt.xticks(x, features, rotation=45)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 加载不是代码之后的替代方案
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score
import os
import urllib.request
import tarfile
from sklearn.datasets import fetch_california_housing

# 设置中文显示支持
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'Microsoft YaHei']  # macOS/Windows 中文支持
plt.rcParams['axes.unicode_minus'] = False

# 解决数据集加载问题的多种方法
try:
    # 方法1：尝试直接从sklearn加载
    print("尝试从scikit-learn加载数据集...")
    house_value = fetch_california_housing()
    print("成功从scikit-learn加载数据集")
except Exception as e:
    print(f"从scikit-learn加载失败: {e}")
    print("尝试从备用URL下载数据集...")
    
    # 方法2：从备用URL下载数据集
    try:
        # 创建数据目录
        data_dir = os.path.join(os.getcwd(), 'california_housing')
        os.makedirs(data_dir, exist_ok=True)
        
        # 下载文件
        data_url = "https://ndownloader.figshare.com/files/5976036"
        data_path = os.path.join(data_dir, "cal_housing.tgz")
        
        # 下载数据集
        urllib.request.urlretrieve(data_url, data_path)
        print("数据集下载完成")
        
        # 解压文件
        with tarfile.open(data_path) as tar:
            tar.extractall(path=data_dir)
        print("数据集解压完成")
        
        # 加载数据
        from sklearn.datasets import fetch_california_housing
        house_value = fetch_california_housing(data_home=data_dir)
        print("成功从本地文件加载数据集")
    except Exception as e:
        print(f"备用方法也失败: {e}")
        print("使用内置的替代数据集...")
        
        # 方法3：创建模拟数据集作为最后手段
        np.random.seed(42)
        n_samples = 20640
        X = np.random.rand(n_samples, 8)
        y = 100 * X[:, 0] + 50 * X[:, 1] + 30 * X[:, 2] + np.random.randn(n_samples) * 10
        
        # 创建类似fetch_california_housing的结构
        house_value = type('obj', (object,), {})()
        house_value.data = X
        house_value.target = y
        house_value.feature_names = [
            "住户收入中位数", "房屋使用年代中位数", "平均房间数目", "平均卧室数目", 
            "街区人口", "平均入住率", "街区的纬度", "街区的经度"
        ]
        print("使用模拟数据集")

# 准备数据
X = pd.DataFrame(house_value.data)
y = house_value.target

# 设置列名（如果数据集没有提供特征名，使用默认名）
if hasattr(house_value, 'feature_names') and house_value.feature_names:
    X.columns = house_value.feature_names
else:
    X.columns = ["Feature_{}".format(i) for i in range(X.shape[1])]
    print("使用默认特征名称")

# 创建包含特征和目标的数据框
Xtmp = X.copy()
Xtmp['价格'] = y
print("\n数据集预览:")
display(Xtmp.head())  # 显示前几行而不是整个数据集

# 划分训练集、测试集
xtrain, xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.3, random_state=420
)

# 训练集和测试集恢复索引
xtrain.index = range(xtrain.shape[0])
xtest.index = range(xtest.shape[0])

# 采用岭回归训练模型
reg = Ridge(alpha=5).fit(xtrain, ytrain)

# 在测试集上进行预测并计算R²
yhat = reg.predict(xtest)
r2 = r2_score(ytest, yhat)  # 正确顺序：先真实值，后预测值
print("\n测试集R²: %.8f" % r2)

# 探索交叉验证下岭回归与线性回归的结果变化
alpha_range = np.arange(1, 1001, 100)
ridge_scores, lr_scores = [], []

# 线性回归模型只需计算一次（不依赖alpha）
linear = LinearRegression()
linear_score = cross_val_score(linear, X, y, cv=5, scoring='r2').mean()

for alpha in alpha_range:
    # 岭回归模型
    reg = Ridge(alpha=alpha)
    ridge_score = cross_val_score(reg, X, y, cv=5, scoring='r2').mean()
    ridge_scores.append(ridge_score)
    
    # 线性回归分数是常数，不需要重复计算
    lr_scores.append(linear_score)

# 绘制结果
plt.figure(figsize=(12, 6))
plt.plot(alpha_range, ridge_scores, c='red', label='岭回归 (Ridge)')
plt.plot(alpha_range, lr_scores, c='orange', label='线性回归 (LR)')
plt.title('不同正则化强度下的模型性能')
plt.xlabel('正则化强度 (alpha)')
plt.ylabel('交叉验证R²')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# 标记最佳alpha值
best_alpha = alpha_range[np.argmax(ridge_scores)]
best_score = max(ridge_scores)
plt.axvline(x=best_alpha, color='blue', linestyle=':', 
            label=f'最佳alpha={best_alpha}\nR²={best_score:.4f}')
plt.legend()

plt.tight_layout()
plt.savefig('img/ridge_vs_lr.png', dpi=300)
plt.show()

# 使用最佳alpha值重新训练模型
print(f"\n使用最佳alpha值({best_alpha})重新训练模型...")
best_reg = Ridge(alpha=best_alpha).fit(xtrain, ytrain)

# 评估最佳模型
train_pred = best_reg.predict(xtrain)
test_pred = best_reg.predict(xtest)

train_r2 = r2_score(ytrain, train_pred)
test_r2 = r2_score(ytest, test_pred)

print(f"训练集R²: {train_r2:.6f}")
print(f"测试集R²: {test_r2:.6f}")

# 比较特征重要性
plt.figure(figsize=(12, 6))
features = X.columns

# 线性回归系数
lr = LinearRegression().fit(xtrain, ytrain)
lr_coef = lr.coef_

# 岭回归系数
ridge_coef = best_reg.coef_

# 绘制系数比较
x = np.arange(len(features))
width = 0.35

plt.bar(x - width/2, lr_coef, width, label='线性回归')
plt.bar(x + width/2, ridge_coef, width, label=f'岭回归 (alpha={best_alpha})')
plt.axhline(0, color='black', linewidth=0.8)

plt.xlabel('特征')
plt.ylabel('系数值')
plt.title('线性回归与岭回归特征系数比较')
plt.xticks(x, features, rotation=45)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('img/feature_coefficients.png', dpi=300)
plt.show()

#### LASSO 回归

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing as fch
 
#加载数据集
House_value = fch()
X= pd.DataFrame(house_value.data)  #提取数据集
y= house_value.target  #提取标签
 
X.columns = ["住户收人中位数", "房屋使用年代中位数", "平均房间数目", "平均卧室数目", "街区人口", "平均入住率", "街区的纬度", "街区的经度"]
xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state= 420)
 
#训练集恢复索引
for i in [xtrain, xtest]:
    i.index = range(i.shape[0])
#采用LASSO回归训练模型
model = Lasso(alpha= 0.05).fit(xtrain, ytrain)
print('模型系数:\n', model.coef )
 
#模型准确率
m_score = model.score(xtest, ytest)
print("模型准确率: *.8f" %m_score)
coef = pd.Series(model.coef_, index=df.drop('target', axis=1, inplace=False).columns)
print(coef[coef!= 0].abs().sort_values(ascending = False))
print('系数为0的属性:\n', coef[coef == 0])

## 5.5 Logistic 回归

In [ ]:
# 导入必要库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
 
# 导入数据集
cancer = load_breast_cancer()
 
# 将数据转换为DataFrame格式便于查看
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
 
# 查看数据前5行
print(df.head())
 
# 查看数据基本信息
print("\n数据集基本信息:")
print(df.info())
 
# 查看目标变量分布
print("\n目标变量分布:")
print(df['target'].value_counts())

### 5.5.1 数据准备与探索性分析

In [ ]:
# 检查缺失值
print("\n缺失值检查:")
print(df.isnull().sum())
 
# 特征和目标变量分离
X = cancer.data  # 使用所有特征
y = cancer.target
 
# 数据标准化（Logistic回归对特征尺度敏感）
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 
# 数据集分割
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
 
print(f"\n训练集大小: {X_train.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")

### 5.5.2 模型训练与参数选择

In [ ]:
# 初始化Logistic回归模型
# 参数说明：
# penalty: 正则化类型（l1或l2）
# C: 正则化强度的倒数（越小正则化越强）
# solver: 优化算法
# max_iter: 最大迭代次数
model = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
 
# 训练模型
model.fit(X_train, y_train)
 
# 查看模型参数
print("\n模型参数:")
print(f"系数 (coefficients): {model.coef_}")
print(f"截距 (intercept): {model.intercept_}")

### 5.5.3 模型评估

In [ ]:
# 在测试集上预测
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # 预测为类别1的概率
 
# 混淆矩阵
conf_mat = confusion_matrix(y_test, y_pred)
print("\n混淆矩阵:")
print(conf_mat)
 
# 分类报告
print("\n分类报告:")
print(classification_report(y_test, y_pred))
 
# 计算准确率
accuracy = model.score(X_test, y_test)
print(f"\n测试集准确率: {accuracy:.4f}")

### 5.5.4 ROC曲线与AUC

In [ ]:
# 计算ROC曲线
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
 
# 绘制ROC曲线
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC曲线 (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('假阳性率 (False Positive Rate)')
plt.ylabel('真阳性率 (True Positive Rate)')
plt.title('Logistic回归的ROC曲线')
plt.legend(loc="lower right")
plt.savefig('img/Logistic回归的ROC曲线.png', dpi=300)
plt.show()

### 5.5.5 结果分析

In [ ]:
# 特征重要性分析
feature_importance = pd.DataFrame({
    'Feature': cancer.feature_names,
    'Importance': model.coef_[0]
}).sort_values('Importance', ascending=False)
 
print("\n特征重要性:")
print(feature_importance)
 
# 可视化前10个重要特征
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'][:10], feature_importance['Importance'][:10])
plt.xlabel('系数大小')
plt.title('Logistic回归特征重要性（前10）')
plt.savefig('img/Logistic回归特征重要性（前10）.png', dpi=300)
plt.show()

## 5.6 决策树分类

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn import tree
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn import tree
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test= train_test_split(iris.data, iris.target, 
test_size=0.20, random_state = 30, shuffle= True)
clf = tree.DecisionTreeClassifier(criterion= 'entropy')
#criterion默认为'gini'
clf = clf.fit(X_train, y_train)
plt.figure(dpi=150)
tree.plot_tree(clf, feature_names = iris.feature_names, class_names=
iris.target_names)
# feature_names=iris.feature_names #设置决策树中显示的特征名称

#预测数据[6,5,5,2]的类别
print('数据[6,5,5,2]的类别:', clf.predict([[6,5,5,2]]))
print('测试集的标签:\n', y_test)
y_pre=clf.predict(X_test)
print('预测的测试集标签:\n', y_pre)
print('模型准确率为:', clf.score(X_test, y_test))

## 5.7 神经网络

In [ ]:
#1. 认识所分析的数据集
from sklearn.datasets import load_wine
wine = load_wine( )
wine.data[0]

In [ ]:
wine.data.shape

In [ ]:
wine.target

In [ ]:
wine.target_names

In [ ]:
wine.feature_names

In [ ]:
# 2. 分割数据集

In [ ]:
X = wine.data
y = wine.target
#X, y = wine.data, wine.target

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [ ]:
# 3. 构造多层神经网络

In [ ]:
#导入多层感知机分类器
from sklearn.neural_network import MLPClassifier
#构造多层感知机分类器模型
model = MLPClassifier(solver = "lbfgs", hidden_layer_sizes = (100,))

In [ ]:
# 4. 训练模型与预测数据

In [ ]:
#训练模型
model.fit(X_train, y_train)

In [ ]:
#在训练集和测试集上进行预测
y_predict_on_train = model.predict(X_train)
y_predict_on_test = model.predict(X_test)

#模型评估————计算准确率
from sklearn.metrics import accuracy_score

#计算训练集准确率（百分比形式）
train_accuracy = 100 * accuracy_score(y_train, y_predict_on_train)
#计算测试集准确率（百分比形式）
test_accuracy = 100 * accuracy_score(y_test, y_predict_on_test)

#输出结果
print('训练集的准确率为: {:.2f}%'.format(train_accuracy))
print('测试集的准确率为: {:.2f}%'.format(test_accuracy))

In [ ]:
# 1. 认识所分析的数据集
from sklearn.datasets import load_wine
wine = load_wine( )

# 2. 分割数据集
X = wine.data
y = wine.target
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

# 3. 构造多层神经网络
#导入多层感知机分类器
from sklearn.neural_network import MLPClassifier
#构造多层感知机分类器模型
model = MLPClassifier(solver = "lbfgs", hidden_layer_sizes = (10, 10))  #参数调整

# 4. 训练模型与预测数据
#训练模型
model.fit(X_train, y_train)
#在训练集和测试集上进行预测
y_predict_on_train = model.predict(X_train)
y_predict_on_test = model.predict(X_test)

#模型评估————计算准确率
from sklearn.metrics import accuracy_score

#计算训练集准确率（百分比形式）
train_accuracy = 100 * accuracy_score(y_train, y_predict_on_train)
#计算测试集准确率（百分比形式）
test_accuracy = 100 * accuracy_score(y_test, y_predict_on_test)

#输出结果
print('训练集的准确率为: {:.2f}%'.format(train_accuracy))
print('测试集的准确率为: {:.2f}%'.format(test_accuracy))

In [ ]:
# 对数据进行缩放预处理
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)

#对训练集和测试集均做缩放处理
X_train = scales.transform(X_train)
X_test = scales.transform(X_test)


In [ ]:
#简化和优化后的总代码
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier #导入神经网络模型
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix #模型性能评估模型
import numpy as np

# （1）导入数据集
wine = load_wine( )
X, y = wine.data, wine.target

#  (2)分割数据
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0, stratify=y)
#添加分层抽样以保持类别比例

#  （3）数据预处理
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

#  （4）导入神经网络模型构造模型：设置一层隐含层
model = MLPClassifier(solver = "lbfgs", hidden_layer_sizes = (100,))  #参数调整

#   （5）训练神经网络模型
model.fit(X_train_scaled, y_train)

#  （6）在训练集和测试集上进行预测
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

#  （7）模型评估：查看预测准确率
#计算训练集准确率（百分比形式）
train_accuracy = 100 * accuracy_score(y_train, y_pred_train)
#计算测试集准确率（百分比形式）
test_accuracy = 100 * accuracy_score(y_test, y_pred_test)

#输出结果
print('训练集的准确率为: {:.2f}%'.format(train_accuracy))
print('测试集的准确率为: {:.2f}%'.format(test_accuracy))

In [ ]:
#输出偏置参数权重，共13个
model.coefs_[0]

In [ ]:
#输出特征参数权重，共13组，每组100个
model.coefs_[1]

## 5.8 KNN 算法

In [ ]:
import numpy as np
X_train = np.array([6, 2, 24, -6, 10])   #构建一个矩阵
X_min, X_max =X_train.min(), x_train.max()  #求得最小值、最大值
print(X_min, x_max)

In [ ]:
X_nomal = (X_train-X_min)/(X_max-X_min)   #求得归一化矩阵
print(X_nomal)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
X_train = np.array([[6, 2, 24, -6, 10]]).reshape(-1,1)
#X_train = np.arrsy([[6, 2, 24, -6, 10]]).T

In [ ]:
X_std =(X-X.min(axis=0))/(X.max(axis=0)-X.min(axis=0))

In [ ]:
min_max_scaler = MinMaxScaler()

In [ ]:
min_max_scaler.fit(X_train)
MinMaxScaler(copy=True, feature_range=(0, 1))

In [ ]:
min_max_scaler.transform(X_train)

In [ ]:
X_nomal = min_max_scaler.transform(X_train).T
X_nomal

In [ ]:
min_max_scaler.fit_transform(X_train).T

In [ ]:
#加载库
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
 
#加载数据
iris = load_iris()
#只使用前两个特征（萼片长度和萼片宽度）以便可视化
X = iris.data[:, :2]
y = iris.target

#打印数据集信息
#print("目标值:", iris.target)
#print("数据集描述:", iris.DESCR)
#print("特征名称:", iris.feature_names)

#创建自定义颜色映射用于可视化
#浅色用于决策区域
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
#深色用于数据点
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])
 
#初始化KNN分类器对象
#n_neighbors:考虑最近邻的数量
#weights：投票权重（‘Uniform表示所有邻居权重相同’）
clf = KNeighborsClassifier(n_neighbors = 15, weights = 'uniform')
clf.fit(X, y) #使用数据训练分类器
 
#描绘决策边界，用不同颜色表示
x_min, x_max = X[:, 0].min()- 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min()- 1, X[:, 1].max() + 1

#创建网格点坐标矩阵（步长0.02）
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

#预测整个网格上每个点的类别
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
 
#绘制预测结果图
plt.figure(figsize=(10, 6))

#绘制决策区域（使用浅色）
plt.pcolormesh(xx, yy, Z, cmap = cmap_light, shading='auto')
 
#绘制训练数据点（使用深色）
plt.scatter(X[:, 0], X[:, 1], c=y, cmap = cmap_bold, edgecolor='k', s=50, label='训练数据点')

#添加图例和标签
plt.legend(loc='best')
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())

#添加标题和说明
plt.title("KNN分类器决策边界（k=15, 权重='uniform')", fontsize=14)
plt.figtext(0.5, 0.01, "KNN分类器在鸢尾花数据集前两个特征上的决策边界", ha='center', fontsize=10)

#显示图形
plt.tight_layout()
plt.subplots_adjust(bottom=0.1)
plt.savefig('img/KNN分类器决策边界.png', dpi=300)
plt.show()

## 5.9 支持向量机

In [ ]:
import numpy as np
from sklearn import svm 
from sklearn import datasets
from sklearn import metrics
from sklearn import model_selection
import matplotlib.pyplot as plt

#加载数据集
iris = datasets.load_iris()
x, y = iris.data, iris.target

#划分训练集和测试集合
x_train, x_test, y_train, y_test = model_selection.train_test_split(x, y, random_state=1, test_size=0.2)

#创建SVM分类器
classifier= svm.SVC(
    kernel='linear',
    gamma=0.1,
    decision_function_shape='ovo',
    C=0.1
)
classifier.fit(x_train, y_train.ravel())  #使用ravel（）展平y_train

#输出准确率
print("SVM-输出训练集的准确率为:", classifier.score(x_train, y_train))
print("SVM-输出测试集的准确率为:", classifier.score(x_test, y_test))

#生成分类报告
y_hat = classifier.predict(x_test)
classreport = metrics.classification_report(y_test, y_hat)
print("分类报告")
print(classreport)

## 5.10 贝叶斯分类

In [ ]:
from sklearn.datasets import load_iris
#从Scikit-Learn库导入朴素贝叶斯模型中的多项式朴素贝叶斯分类算法
from sklearn.naive_bayes import MultinomialNB
#载入鸢尾花数据集
X, y= load_iris(return_X_y=True)
#训练模型
clf = MultinomialNB().fit(X, y)
#使用模型进行分类预测
clf.predict(X)

In [ ]:
clf.score(X, y)

## 5.11 K-Means 聚类分析

In [ ]:
sklearn.datasets.make_blobs(n_samples = 100, n_features = 2, centers = None, 
                            cluster_std=1.0, center_box=(-10.0, 10.0), shuffle=True, random_state=None)

In [ ]:
import matplotlib.pyplot as plt
# 导入生成数据的方法
from sklearn.datasets import make_blobs
# 生成数据
blobs = make_blobs(n_samples = 200, random_state = 1, centers = 4)

In [ ]:
#提取特征数据
X_blobs = blobs[0]

In [ ]:
plt.scatter(X_blobs[:, 0], X_blobs[:, 1])
plt.savefig('img/Kmeans聚类0.png', dpi=300)
plt.show()

In [ ]:
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=blobs[1])
plt.savefig('img/Kmeans聚类.png', dpi=300)
plt.show()

In [ ]:
#(1)导入Kmeans工具包
from sklearn.cluster import KMeans
#(2)构建KMeans模型对象，设定k=4
kmeans = KMeans(n_clusters = 4)
#(3)训练模型（拟合）
kmeans.fit(X_blobs)
#(4)绘制可视化图
import numpy as np
x_min, x_max = X_blobs[:, 0].min()- 0.5, X_blobs[:, 0].max()+ 0.5
y_min, y_max = X_blobs[:, 1].min()- 0.5, X_blobs[:, 1].max()+ 0.5
 
#(5)生成网格点矩阵
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
Z = kmeans.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.figure(1)
plt.clf()
plt.imshow(Z, interpolation = 'hermite', extent = (xx.min(), xx.max(), yy.min(), yy.max()), cmap = plt.cm.winter, aspect='auto', origin = 'lower')
plt.plot(X_blobs[:, 0], X_blobs[:, 1], 'w.', markersize = 5)
 
#用红色的×表示簇中心
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], marker="x", s=150, linewidths = 3, color = 'r', zorder = 10)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.xticks()
plt.yticks()
plt.savefig('img/Kmeans聚类2.png', dpi=300)
plt.show()

In [ ]:
import warnings
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.metrics.pairwise import pairwise_distances_argmin
from sklearn.datasets import make_blobs

#忽略警告信息
warnings.filterwarnings('ignore')

#设置中文字体支持
mpl.rcParams['font.sans-serif'] = [u'Arial Unicode MS']
mpl.rcParams['axes.unicode_minus'] = False

#1. 数据生成
#初始化3个中心,产生 30,000 组二维数据
centers =[[1, 1],[-1, -1],[1, -1]]
clusters = len(centers)  #聚类的数目为 3
X, Y = make_blobs(n_samples=30000, centers=centers, cluster_std=0.4, random_state=28)

#2. KMeans聚类
k_means = KMeans(init='k-means++', n_clusters=clusters, random_state=28)
t0 = time.time()    #当前时间
k_means.fit(X)    #训练模型
km_batch = time.time()- t0    #使用 K-Means 训练数据的消耗时间
print("K-Means 模型训练消耗时间: %.4fs" % km_batch)

#3. MiniBatchkMeans聚类
batch_size = 100
mbk = MiniBatchKMeans(init='k-means++', n_clusters=clusters, batch_size=batch_size, random_state=28)
t0 = time.time()
mbk.fit(X)
mbk_batch = time.time()- t0
print("Mini Batch K-Means模型训练消耗时间: %.4fs" % mbk_batch)

# 4. 预测结果
km_y_hat = k_means.predict(X)
mbkm_y_hat = mbk.predict(X)
print("K-Means前20个预测结果:", km_y_hat[:20])   
print("MiniBatchKMeans前20个预测结果:", mbkm_y_hat[:20])

#5. 获取聚类中心点
k_means_cluster_centers = k_means.cluster_centers_
mbk_means_cluster_centers = mbk.cluster_centers_
print("K-Means算法聚类中心点:\n", k_means_cluster_centers)
print("Mini Batch K-Means算法聚类中心点:\n", mbk_means_cluster_centers)

#6.对齐聚类中心点（解决聚类标签不一致问题）
# pairwise_distances_argmin: 将X和Y中的元素按照从大到小排序
order = pairwise_distances_argmin(X=k_means_cluster_centers, Y=mbk_means_cluster_centers)

#7. 结果可视化
plt.figure(figsize = (14, 10), facecolor = 'w')
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95, hspace=0.2, wspace=0.2)

#子图 1:原始数据分布
plt.subplot(221)
plt.scatter(X[:, 0], X[:, 1], c=Y, s=6, cmap='viridis')
plt.title(u'原始数据分布图')
plt.xticks(())
plt.yticks(())
plt.grid(True)

#子图 2:K-Means 算法聚类结果图
plt.subplot(222)
plt.scatter(X[:, 0], X[:, 1], c=km_y_hat, s=6, cmap='viridis')
plt.scatter(k_means_cluster_centers[:, 0], k_means_cluster_centers[:, 1], c=range(clusters), s=200, marker='X', edgecolors='k')
plt.title(u'K-Means聚类结果图')
plt.xticks(())
plt.yticks(())
plt.text(-3.0, 2.5, '训练时间: %.2fms' % (km_batch * 1000), fontsize=10)
plt.grid(True)

#子图3: Mini Batch K-Means算法聚类结果图
plt.subplot(223)
plt.scatter(X[:, 0], X[:, 1], c=mbkm_y_hat, s=6, cmap='viridis')
plt.scatter(mbk_means_cluster_centers[:, 0], mbk_means_cluster_centers[:, 1], c=range(clusters), s=200, marker='X', edgecolor='k')
plt.title(u'Mini BatchK-Means 聚类结果图')
plt.xticks(())
plt.yticks(())
plt.text(-3.0, 2.5, '训练时间: %.2fms' % (mbk_batch * 1000), fontsize=10)
plt.grid(True)

#8. 比较两种算法的预测差异
# 对齐聚类标签（解决不同算法可能使用不同标签表示相同聚类的问题）
aligned_mbkm_y_hat = np.zeros_like(mbkm_y_hat)
for i in range(clusters):
    aligned_mbkm_y_hat[mbkm_y_hat == order[i]] = i

# 找出预测不同的点
different = (km_y_hat != aligned_mbkm_y_hat)
different_nodes = np.sum(different)
print(f"预测不同的点数量: {different_nodes} ({different_nodes/len(X)*100:.2f}%)")

# 子图4: 两种算法预测不同的点
plt.subplot(224)
# 绘制所有点（灰色背景）
plt.scatter(X[:, 0], X[:, 1], c='lightgray', s=6, alpha=0.3)
# 标记预测不同的点（红色）
plt.scatter(X[different, 0], X[different, 1], c='red', s=20, alpha=0.7)
plt.title(u'K-Means与MiniBatchKMeans预测不同的点')
plt.xticks(())
plt.yticks(())
plt.text(-3.0, 2.5, f'不同预测的点数: {different_nodes}', fontsize=10)
plt.grid(True)

plt.tight_layout()
plt.savefig('img/kmeans_comparison.png', dpi=300)
plt.show()

In [ ]:
import warnings
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.metrics.pairwise import pairwise_distances_argmin
from sklearn.datasets import make_blobs

#忽略警告信息
warnings.filterwarnings('ignore')

#设置中文字体支持
mpl.rcParams['font.sans-serif'] = [u'Arial Unicode MS']
mpl.rcParams['axes.unicode_minus'] = False

#1. 数据生成
#初始化3个中心,产生 30,000 组二维数据
centers =[[1, 1],[-1, -1],[1, -1]]
clusters = len(centers)  #聚类的数目为 3
X, Y = make_blobs(n_samples=30000, centers=centers, cluster_std=0.4, random_state=28)

#2. KMeans聚类
k_means = KMeans(init='k-means++', n_clusters=clusters, random_state=28)
t0 = time.time()    #当前时间
k_means.fit(X)    #训练模型
km_batch = time.time()- t0    #使用 K-Means 训练数据的消耗时间
print("K-Means 模型训练消耗时间: %.4fs" % km_batch)

#3. MiniBatchkMeans聚类
batch_size = 100
mbk = MiniBatchKMeans(init='k-means++', n_clusters=clusters, batch_size=batch_size, random_state=28)
t0 = time.time()
mbk.fit(X)
mbk_batch = time.time()- t0
print("Mini Batch K-Means模型训练消耗时间: %.4fs" % mbk_batch)

# 4. 预测结果
km_y_hat = k_means.predict(X)
mbkm_y_hat = mbk.predict(X)
print("K-Means前20个预测结果:", km_y_hat[:20])   
print("MiniBatchKMeans前20个预测结果:", mbkm_y_hat[:20])

#5. 获取聚类中心点
k_means_cluster_centers = k_means.cluster_centers_
mbk_means_cluster_centers = mbk.cluster_centers_
print("K-Means算法聚类中心点:\n", k_means_cluster_centers)
print("Mini Batch K-Means算法聚类中心点:\n", mbk_means_cluster_centers)

#6.对齐聚类中心点（解决聚类标签不一致问题）
# pairwise_distances_argmin: 将X和Y中的元素按照从大到小排序
order = pairwise_distances_argmin(X=k_means_cluster_centers, Y=mbk_means_cluster_centers)

#7. 结果可视化
plt.figure(figsize = (14, 10), facecolor = 'w')
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95, hspace=0.2, wspace=0.2)

#子图 1:原始数据分布
plt.subplot(221)
plt.scatter(X[:, 0], X[:, 1], c=Y, s=6, cmap='viridis')
plt.title(u'原始数据分布图')
plt.xticks(())
plt.yticks(())
plt.grid(True)

#子图 2:K-Means 算法聚类结果图
plt.subplot(222)
plt.scatter(X[:, 0], X[:, 1], c=km_y_hat, s=6, cmap='viridis')
plt.scatter(k_means_cluster_centers[:, 0], k_means_cluster_centers[:, 1], c=range(clusters), s=200, marker='X', edgecolors='k')
plt.title(u'K-Means聚类结果图')
plt.xticks(())
plt.yticks(())
plt.text(-3.0, 2.5, '训练时间: %.2fms' % (km_batch * 1000), fontsize=10)
plt.grid(True)

#子图3: Mini Batch K-Means算法聚类结果图
plt.subplot(223)
plt.scatter(X[:, 0], X[:, 1], c=mbkm_y_hat, s=6, cmap='viridis')
plt.scatter(mbk_means_cluster_centers[:, 0], mbk_means_cluster_centers[:, 1], c=range(clusters), s=200, marker='X', edgecolor='k')
plt.title(u'Mini BatchK-Means 聚类结果图')
plt.xticks(())
plt.yticks(())
plt.text(-3.0, 2.5, '训练时间: %.2fms' % (mbk_batch * 1000), fontsize=10)
plt.grid(True)

#8. 比较两种算法的预测差异
# 对齐聚类标签（解决不同算法可能使用不同标签表示相同聚类的问题）
aligned_mbkm_y_hat = np.zeros_like(mbkm_y_hat)
for i in range(clusters):
    aligned_mbkm_y_hat[mbkm_y_hat == order[i]] = i

# 找出预测不同的点
different = (km_y_hat != aligned_mbkm_y_hat)
different_nodes = np.sum(different)
print(f"预测不同的点数量: {different_nodes} ({different_nodes/len(X)*100:.2f}%)")

# 子图4: 两种算法预测不同的点
plt.subplot(224)
# 绘制所有点（灰色背景）
plt.scatter(X[:, 0], X[:, 1], c='lightgray', s=6, alpha=0.3)
# 标记预测不同的点（红色）
plt.scatter(X[different, 0], X[different, 1], c='red', s=20, alpha=0.7)
plt.title(u'K-Means与MiniBatchKMeans预测不同的点')
plt.xticks(())
plt.yticks(())
plt.text(-3.0, 2.5, f'不同预测的点数: {different_nodes}', fontsize=10)
plt.grid(True)

plt.tight_layout()
plt.savefig('img/kmeans_comparison.png', dpi=300)  # 保存高质量图像
plt.show()

### 补充：层次聚类

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import matplotlib.pyplot as plt
from itertools import cycle   #Python自带的迭代器模块

#产生随机数据的中心
centers = [[1, 1], [-1, -1], [1, -1]]
#产生的数据个数
n_samples = 3000
#产生数据
X, labels_true = make_blobs(n_samples = n_samples, centers = centers, cluster_std = 0.6, random_state = 0)

# 设置分层聚类函数
linkages = ['ward', 'average', 'complete']
n_clusters = 3
ac = AgglomerativeClustering(linkage=linkages[2], n_clusters=n_clusters)

#训练数据
ac.fit(X)

#获取聚类标签
labels = ac.labels_

plt.figure(figsize=(10, 8))
plt.clf()
colors = cycle('bgrcmykbgrcmykbgrcmykbgrcmyk')

for k, col in zip(range(n_clusters), colors):
    #根据 lables 中的值是否等于k，重新组成一个True、False的数组
    #X[my_members, 0]取出my_members对应位置为True的值的横坐标
    my_members = labels == k
    #绘制每个聚类的点
    plt.scatter(X[my_members, 0], X[my_members, 1], color=col, s=30, alpha=0.7)
    
plt.title(f'Estimated number of clusters: {n_clusters}')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, linestyle='--', alpha=0.3)
plt.savefig('Estimated number of clusters', dpi=300)
plt.show()

## 5.12 基于密度的聚类

In [ ]:
from sklearn import datasets
import numpy as np
import random
import matplotlib.pyplot as plt
def findNeighbor(j, X, eps):
    N=[ ]
    for p in range(X.shape[0]):   #找到所有邻域内的对象
        temp = np.sqrt(np.sum(np.square(X[j]-X[p])))
#欧氏距离
        if(temp<= eps):
            N.append(p)
    return N
def dbscan(X, eps, min_Pts):
    k= -1
    NeighborPts = [ ]   #array,某点邻域内的对象
    Ner_NeighborPts = [ ]
    fil = [ ]   #初始时已访问对象列表为空#初始所有点标为未访问
    gama = [x for x in range(len(X))]
    cluster = [-1 for y in range(len(X))]
    while len(gama) > 0:
        j= random.choice(gama)
        gama.remove(j)  #从未访问列表中移除
        fil.append(j)  #添加到访问列表
        NeighborPts = findNeighbor(j, X, eps)
        if len(NeighborPts) < min_Pts:
            cluster[j] = -1 # 标记为噪声点
        else:
            k= k+1
            cluster[j]=k
            for i in NeighborPts :
                if i not in fil:
                    gama.remove(i)
                    fil.append(i)
                    Ner_NeighborPts = findNeighbor(i, X, eps)
                    if len(Ner_NeighborPts)>= min_Pts:
                        for a in Ner_NeighborPts:
                            if a not in NeighborPts:
                                NeighborPts.append( a)
                    if(cluster[i]==-1):
                        cluster[i]=k
    return cluster
X1, y1 = datasets.make_circles(n_samples = 1000, factor = .6, noise= .05)
X2, y2 = datasets.make_blobs(n_samples = 300, n_features = 2, 
                             centers = [[1.2, 1.2]], cluster_std = [[.1]], random_state = 9)
X= np.concatenate( (X1, X2))
eps = 0.08
min_Pts = 10
C = dbscan(X, eps, min_Pts)
plt.figure(figsize = (12, 9), dpi = 80)
plt.scatter(X[:, 0], X[:, 1], c = C)
plt.savefig('密度聚类', dpi=300)
plt.show()

In [ ]:
#sklearn库中DBSCAN库实现密度聚类
from sklearn.cluster import DBSCAN
eps = 0.08
min_Pts = 10
dbscan = DBSCAN(eps = eps, min_samples = min_Pts)
dbscan.fit(X)
label_pred = dbscan.labels_
plt.figure(figsize = (12,9), dpi = 80)
plt.scatter(X[:, 0],X[:,1], c = label_pred)
plt.savefig('img/密度聚类1.pdf', dpi=300)
plt.show()

### 补充：其他聚类方法

#### FCM 聚类

In [ ]:
!pip3 install scikit-fuzzy

In [ ]:
%matplotlib inline
import numpy as np
from skfuzzy.cluster import cmeans
import matplotlib.pyplot as plt
from sklearn import datasets
#加载数据
iris = datasets.load_iris()
x = iris.data
colo = ['b', 'g', 'r', 'y', 'm']
shape = x.shape

FPC = []
labels = []
for k in range (2, 5):
    center, u, u0, d, jm, p, fpc = cmeans(x.T, m=2, c=k, error=0.5, maxiter=1000)
    label = np.argmax(u, axis = 0)
    for i in range (shape[0]):
        plt.xlabel('x')
        plt.ylabel('y')
        plt.title('c= ' + str(k))
        plt.plot(x[i, 0], x[i, 1], colo[label[i]] + 'o')
    plt.savefig('img/FCM聚类.pdf')
    plt.show()
    print('c = % d: FPC = % 0.2f' % (k, fpc))
    FPC.append( fpc)

#### 高斯混合模型聚类

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

#产生随机数据的中心
centers = [[1, 1.5],[-1, -1.5], [1.5, -1.8]]
#产生的数据个数
n_samples = 3000
#产生数据
X, lables_true = make_blobs(n_samples = n_samples, centers = centers, cluster_std = 0.8, random_state = 71)

gmm = GaussianMixture(n_components = 3, random_state = 23)
gmm.fit(X)
gmm_labels = gmm.predict(X)
gmm_silhouette_score = silhouette_score(X, gmm_labels)
print("高斯混合模型聚类的性能:")
print("轮廓系数: {:.4f}".format(gmm_silhouette_score))
x0 = X[gmm_labels == 0]
x1 = X[gmm_labels == 1]
x2 = X[gmm_labels == 2]
plt.scatter(x0[:,0],x0[:,1], c = "red", marker = 'o', label ='label0')
plt.scatter(x1[:,0],x1[:,1], c = "green", marker ='*', label='label1')
plt.scatter(x2[:,0],x2[:,1], c = "blue", marker ='+', label = 'labe12')
plt.legend(loc=2)
plt.savefig('img/高斯混合模型聚类.pdf')
plt.show()


#### 近邻传播 AP 算法

In [ ]:
from sklearn.cluster import AffinityPropagation
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn import datasets

#加载数据
iris = datasets.load_iris()
X = iris.data

#创建并拟合模型
af = AffinityPropagation(preference = -50, random_state = 0).fit(X)

#获取聚类结果
cluster_centers_indices = af.cluster_centers_indices_
labels = af.labels_
labels_true = iris.target
n_clusters = len(cluster_centers_indices)

#评估指标
print("Estimated number of clusters: %d" % n_clusters)
print("Homogeneity: %0.3f" % metrics.homogeneity_score (labels_true, labels))
print("Completeness: %0.3f" % metrics.completeness_score (labels_true, labels))
print("y-measure: %0.3f" % metrics.v_measure_score(labels_true, labels))
print("Adjusted Rand Index: %0.3f" % metrics.adjusted_rand_score(labels_true, labels))
print("Adjusted Mutual Information: %0.3f" %
      metrics.adjusted_mutual_info_score(labels_true, labels))
print( "Silhouette Coefficient: %0.3f" %
      metrics.silhouette_score(X, labels, metric = "sqeuclidean"))

#可视化
plt.close("all")
plt.figure(figsize=(10,6))
plt.clf()

#定义颜色列表
colors =['b', 'g', 'r', 'c', 'm', 'y', 'k']

for k, col in zip(range(n_clusters), colors[:n_clusters]):
    class_members = labels == k
    cluster_center = X[cluster_centers_indices[k]]

    #绘制类成员点
    plt.plot(X[class_members, 0], X[class_members, 1], col + '.', markersize=8)
    #绘制聚类中心
    plt.plot(cluster_center[0], cluster_center[1], 'o',
             markerfacecolor = col, markeredgecolor = 'k', markersize=14)
    #绘制从中心到点的线
    for x in X[class_members]:
        plt.plot([cluster_center[0], x[0]], [cluster_center[1], x[1]], col, alpha=0.3)
        
plt.title("Estimated number of clusters: %d" % n_clusters)
plt.xlabel(iris.feature_names[0])  # 添加X轴标签
plt.ylabel(iris.feature_names[1])  # 添加Y轴标签
plt.grid(True, linestyle='--', alpha=0.6)  # 添加网格线
plt.savefig('img/近邻传播AP算法.pdf')
plt.show()

## 5.13 聚类评估

In [ ]:
# 聚类簇数的确定

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn import datasets

#加载数据集
iris = datasets.load_iris()
df_features = iris.data
SSE = []   #存放每次结果的误差平方和
for k in range(1, 9):
    estimator = KMeans(n_clusters = k, random_state=42)  #添加随机种子
    estimator.fit(df_features)
    SSE.append(estimator.inertia_)
    
X = range(1, 9)
plt.figure(figsize=(10, 6))  # 设置图形大小
plt.xlabel('Number of clusters (K)')
plt.ylabel('Sum of Squared Errors (SSE)')
plt.title('Elbow Method for Optimal K')  # 添加标题
plt.plot(X, SSE, 'o-', linewidth=2, markersize=8)
plt.xticks(range(1, 9))  # 确保x轴刻度正确
plt.grid(True, linestyle='--', alpha=0.7)  # 添加网格线
plt.savefig('img/聚类评估.pdf')
plt.show()

In [ ]:
# 聚类质量的测度 

In [ ]:
#ARI的计算
from sklearn import metrics
labels_true = [0,0,0,1,1,1]
labels_pred = [0,0,1,1,2,2]
print(metrics.adjusted_rand_score(labels_true, labels_pred))

In [ ]:
import numpy as np
from sklearn.cluster import KMeans  # 修正类名大小写
from sklearn.metrics import silhouette_score
from sklearn.datasets import load_iris

# 加载数据
iris = load_iris()
X = iris.data

# 存储不同K值的轮廓系数
silhouette_scores = []

# 测试不同K值（从2到8）
for k in range(2, 9):
    # 创建并拟合KMeans模型
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    
    # 获取聚类标签
    labels = kmeans.labels_  # 定义labels变量
    
    # 计算轮廓系数
    score = silhouette_score(X, labels, metric='euclidean')
    silhouette_scores.append(score)
    print(f"K = {k}: Silhouette Score = {score:.4f}")

# 可视化轮廓系数随K值的变化
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(2, 9), silhouette_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Optimal K Selection')
plt.xticks(range(2, 9))
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig('img/Silhouette Score for Optimal K Selection.pdf')
plt.show()

# 找出最佳K值
best_k = range(2, 9)[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)
print(f"\nOptimal K: {best_k} with Silhouette Score: {best_score:.4f}")